# Join ADAM + GDACS — Historical National Exposure

Loads national-level population exposure estimates from both the ADAM and GDACS historical notebooks and joins them into a single dataset where each row is one storm × one country.

**Sources:**
- `adam_historical_national_exposure.csv` — WFP ADAM wind exposure at 60 / 90 / 120 km/h thresholds
- `gdacs_historical_national_exposure.csv` — GDACS wind exposure at 34 kt / 64 kt thresholds

**Join key:** all shared metadata columns (everything except population exposure columns)

In [48]:
import pandas as pd
import ocha_stratus as stratus
from dotenv import load_dotenv

load_dotenv()

ADAM_BLOB  = 'ds-cyclone-exposure/adam_historical_national_exposure.csv'
GDACS_BLOB = 'ds-cyclone-exposure/gdacs_historical_national_exposure.csv'
OUTPUT_CSV = 'ds-cyclone-exposure/combined_historical_national_exposure.csv'

## 1. Load both datasets from blob storage

In [49]:
df_adam  = stratus.load_csv_from_blob(ADAM_BLOB)
df_gdacs = stratus.load_csv_from_blob(GDACS_BLOB)

print(f'ADAM rows:  {len(df_adam)},  columns: {list(df_adam.columns)}')
print(f'GDACS rows: {len(df_gdacs)}, columns: {list(df_gdacs.columns)}')

ADAM rows:  194,  columns: ['storm_name', 'event_id', 'episode_id', 'source', 'from_date', 'alert_level', 'iso3', 'country_name', 'pop_120kmh', 'pop_60kmh', 'pop_90kmh', 'season', 'name', 'index', 'sid', 'atcf_id', 'number', 'genesis_basin', 'provisional', 'storm_id']
GDACS rows: 574, columns: ['storm_name', 'event_id', 'episode_id', 'source', 'from_date', 'alert_level', 'iso3', 'country_name', 'pop_34kt', 'pop_64kt', 'season', 'name', 'index', 'sid', 'atcf_id', 'number', 'genesis_basin', 'provisional', 'storm_id']


## 2. Inspect each dataset

In [50]:
df_merged = df_gdacs.merge(df_adam, on=["sid", "iso3", "from_date", "season", "name", "atcf_id", "number", "genesis_basin", "event_id", "episode_id", "country_name", "provisional", "storm_id"], how="outer", suffixes=("_gdacs", "_adam"))
print(len(df_merged))

679


In [ ]:
# Combine split columns into single columns (take non-null value from either source)
df_merged['alert_level'] = df_merged['alert_level_gdacs'].fillna(df_merged['alert_level_adam'])
df_merged['storm_name'] = df_merged['storm_name_gdacs'].fillna(df_merged['storm_name_adam'])
df_merged['source'] = df_merged['source_gdacs'].fillna(df_merged['source_adam'])

# Drop the suffixed columns
df_merged = df_merged.drop(columns=['alert_level_gdacs', 'alert_level_adam', 
                                     'storm_name_gdacs', 'storm_name_adam',
                                     'source_gdacs', 'source_adam'])

print(f"Combined columns, final shape: {df_merged.shape}")
print(f"Columns: {list(df_merged.columns)}")

## 5. Save combined dataset to Azure blob storage

In [51]:
stratus.upload_csv_to_blob(df_merged, OUTPUT_CSV)
print(f'Saved {len(df_merged)} rows to {OUTPUT_CSV}')

Saved 679 rows to ds-cyclone-exposure/combined_historical_national_exposure.csv


## 6. Export to JSON for dashboard

In [52]:
# Export to JSON for the dashboard
import json

# Replace NaN with None for proper JSON serialization
df_json = df_merged.replace({pd.NA: None, float('nan'): None})
df_json = df_json.where(pd.notna(df_json), None)

# Convert DataFrame to records format
data_records = df_json.to_dict(orient='records')

# Write to JSON file
with open('exposure_data.json', 'w') as f:
    json.dump(data_records, f, indent=2)

print(f'Exported {len(data_records)} records to exposure_data.json')

Exported 679 records to exposure_data.json
